<a href="https://colab.research.google.com/github/davidekim/WRAPs/blob/main/mspa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Pipeline example for creating WRAPed MspA**

In [ ]:
#@title **Setup RFdiffusion and pyrosetta** (~5-10min)
%%time
import os, time
import sys
import subprocess

def run_cmd(cmd):
  process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
  for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()
  process.stdout.close()
  process.wait()

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/RosettaCommons/RFdiffusion.git")
  # install dependencies
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
  os.system("pip install biopython==1.81")
  os.system("pip install -U dm-haiku")
  os.system("pip install ml-collections")
  os.system('pip install --upgrade "jax[cuda12_pip]<0.6.0" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html')
  os.system("cd RFdiffusion; pip install -e .")
  # install PyRosetta
  os.system("pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release")
  os.system("pip install py3Dmol")
print()

os.environ["DGLBACKEND"] = "pytorch"
#os.environ["HYDRA_FULL_ERROR"] = '1'
diffusion_script = "RFdiffusion/scripts/run_inference.py"

In [ ]:
#@title **Install RFdiffusion weights** (~1-5min)
%%time
if not os.path.isdir("RFdiffusion/models"):
  print("installing RFdiffusion weights...")
  os.system("cd RFdiffusion; mkdir models && cd models")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/60f09a193fb5e5ccdc4980417708dbab/Complex_Fold_base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/74f51cfb8b440f50d70878e05361d8f0/InpaintSeq_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/76d00716416567174cdb7ca96e208296/InpaintSeq_Fold_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/5532d2e1f3a4738decd58b19d633b3c3/ActiveSite_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/12fc204edeae5b57713c5ad7dcb97d39/Base_epoch8_ckpt.pt")

In [ ]:
#@title **Get ProteinMPNN git repo**
%%time
import os, time
if not os.path.isdir("ProteinMPNN"):
  run_cmd("git clone https://github.com/dauparas/ProteinMPNN.git")

In [ ]:
#@title **Get WRAPs git repo**
%%time
import os, time
if not os.path.isdir("WRAPs"):
  run_cmd("git clone https://github.com/davidekim/WRAPs.git")

In [ ]:
#@title **Add length and secondary structure elements for the WRAP**
import pyrosetta
import pyrosetta.rosetta.core.import_pose as import_pr_pose
from pyrosetta import *
from pyrosetta.rosetta import *
from pyrosetta.rosetta.core import *
pyrosetta.init(" -mute all ")

def ss_Type_lengths(pdb_name):
  DSSP = pyrosetta.rosetta.protocols.moves.DsspMover()

  pose = import_pr_pose.pose_from_file(pdb_name)
  DSSP.apply(pose)
  DSSP_pose_string = []
  SS_type_lengths = []
  current_SS = ''
  helix_counter = 0
  Loop_counter = 0
  Sheet_counter = 0
  for z in range(1, len(pose.sequence())):
    if current_SS == pose.secstruct(z):
      SS_type_lengths[len(SS_type_lengths)-1][1] = SS_type_lengths[len(SS_type_lengths)-1][1] +1
    if current_SS != pose.secstruct(z):
      SS_type_lengths.append([pose.secstruct(z),1])
      if pose.secstruct(z) == "H":
        helix_counter = helix_counter+1
      if pose.secstruct(z) == "L":
        Loop_counter = Loop_counter+1
      if pose.secstruct(z) == "E":
        Sheet_counter = Sheet_counter+1
      current_SS = pose.secstruct(z)
      DSSP_pose_string.append(pose.secstruct(z))
  return SS_type_lengths

pdb_name = 'WRAPs/WRAP_MspA/mspa_cut.pdb'
ss_types = ss_Type_lengths(pdb_name)
ss_types2 = ss_types.copy()
ss_types2.insert(0,['H',60])
ss_types2.insert(1,['L',3])
ss_types2.insert(2,['H',20])
ss_types2.insert(11,['H',60])
ss_types2.insert(12,['L',3])
ss_types2.insert(13,['H',20])
ss_types2.insert(22,['H',60])
ss_types2.insert(23,['L',3])
ss_types2.insert(24,['H',20])
ss_types2.insert(33,['H',60])
ss_types2.insert(34,['L',3])
ss_types2.insert(35,['H',20])
ss_types2.insert(44,['H',60])
ss_types2.insert(45,['L',3])
ss_types2.insert(46,['H',20])
ss_types2.insert(55,['H',60])
ss_types2.insert(56,['L',3])
ss_types2.insert(57,['H',20])
ss_types2.insert(66,['H',60])
ss_types2.insert(67,['L',3])
ss_types2.insert(68,['H',20])
ss_types2.insert(77,['H',60])
ss_types2.insert(78,['L',3])
ss_types2.insert(79,['H',20])

ss_types3 = [['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9]]

In [ ]:
#@title **Specify which elements interact with the target and within the WRAP**
adjacencies = [[1, 2, 3, 43],
 [0, 4, 5, 6],
 [0, 5],
 [0],
 [1],
 [1, 2],
 [1, 7, 8, 9],
 [6, 10, 11, 12],
 [6, 11],
 [6],
 [7],
 [7, 8],
 [7, 13, 14, 15],
 [12, 16, 17, 18],
 [12, 17],
 [12],
 [13],
 [13, 14],
 [13, 19, 20, 21],
 [18, 22, 23, 24],
 [18, 23],
 [18],
 [19],
 [19, 20],
 [19, 25, 26, 27],
 [24, 28, 29, 30],
 [24, 29],
 [24],
 [25],
 [25, 26],
 [25, 31, 32, 33],
 [30, 34, 35, 36],
 [30, 35],
 [30],
 [31],
 [31, 32],
 [31, 37, 38, 39],
 [36, 40, 41, 42],
 [36, 41],
 [36],
 [37],
 [37, 38],
 [37, 43, 44, 45],
 [0, 42, 46, 47],
 [42, 47],
 [42],
 [43],
 [43, 44]]

In [ ]:
#@title **Generate block_adjacency for diffusion**
import torch
import matplotlib.pyplot as plt

adjacency_matrix_length = 1064
tensor_adj_matrix = torch.zeros(adjacency_matrix_length,adjacency_matrix_length)

# Get indices for start and ends of helices
helix_indices = []
current_start_index = 0
current_end_index = -1
for i in range(0, len(ss_types3)):
  current_end_index = current_end_index + ss_types3[i][1]
  if ss_types3[i][0] == 'H':
    helix_indices.append([current_start_index,current_end_index])
  if ss_types3[i][0] == 'E':
    helix_indices.append([current_start_index,current_end_index])
  current_start_index = current_start_index + ss_types3[i][1]

#Generate block adjacency by helices
padded_adj_matrix2=tensor_adj_matrix
current_start_indices = 0
current_end_indices = 0
helix_counter = 0

for i in range(0, len(ss_types3)):
  current_end_indices = current_end_indices + ss_types3[i][1]
  if ss_types3[i][0] == 'H':
    for k in range(0, len(adjacencies[helix_counter])):
      for l in range(helix_indices[helix_counter][0],helix_indices[helix_counter][1]+1):
        for j in range(helix_indices[adjacencies[helix_counter][k]][0],helix_indices[adjacencies[helix_counter][k]][1]+1):
          padded_adj_matrix2[l, j] = 1

    helix_counter = helix_counter+1
  if ss_types3[i][0] == 'E':
    for k in range(0, len(adjacencies[helix_counter])):
      for l in range(helix_indices[helix_counter][0],helix_indices[helix_counter][1]+1):
        for j in range(helix_indices[adjacencies[helix_counter][k]][0],helix_indices[adjacencies[helix_counter][k]][1]+1):
          padded_adj_matrix2[l, j] = 1
    helix_counter = helix_counter+1
  current_start_indices = current_start_indices + ss_types3[i][1]

tensor_ss_strand = torch.zeros(len(padded_adj_matrix2[0]))
start_position = 0
end_position = 0

for i in range(0, len(ss_types3)):
    end_position = end_position + ss_types3[i][1]
    if ss_types3[i][0] == 'H':
        tensor_ss_strand[start_position:end_position] = 0

    if ss_types3[i][0] == 'E':
        tensor_ss_strand[start_position:end_position] = 1

    if ss_types3[i][0] == 'L':
        tensor_ss_strand[start_position:end_position] = 2

    start_position = start_position + ss_types3[i][1]

if not os.path.isdir("mspa_longer"): os.mkdir("mspa_longer")
ss_strand_filepath = 'mspa_longer/mspA_ss.pt'
adj_matrix_filepath = 'mspa_longer/mspA_adj.pt'

torch.save(tensor_ss_strand, ss_strand_filepath)
torch.save(padded_adj_matrix2, adj_matrix_filepath)


plt.imshow(padded_adj_matrix2)

In [ ]:
#@title **Run scaffold-guided RFdiffusion with symmetry** (~30min to many hours)
%%time
num_designs = 5 #@param ["1", "5", "10"] {type:"raw"}

cmd = f"RFdiffusion/scripts/run_inference.py --config-name=symmetry inference.symmetry='C8' inference.output_prefix=mspa/mspa inference.input_pdb=WRAPs/WRAP_MspA/mspa_cut.pdb inference.num_designs={num_designs} contigmap.contigs=['83/A1-50/0 83/B1-50/0 83/C1-50/0 83/D1-50/0 83/E1-50/0 83/F1-50/0 83/G1-50/0 83/H1-50/0'] inference.model_runner=NRBStyleSelfCond denoiser.noise_scale_ca=0.5 denoiser.noise_scale_frame=0.5 scaffoldguided.scaffoldguided=True scaffoldguided.scaffold_dir=mspa_longer diffuser.T=30"
print(cmd)
run_cmd(cmd)


In [ ]:
#@title **Run tied (symmetric) Soluble ProteinMPNN**
%%time

from pyrosetta.rosetta.core.pose import append_pose_to_pose
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.minimization_packing import MinMover

#@markdown Select to minimize MPNN output PDBs to cleanup sidechains (optional for aesthetics)
minimize = False #@param {type:"boolean"}

import glob

chains_to_design = "A B C D E F G H"
designsdir = "mspa"
path_for_parsed_chains=f"./{designsdir}/parsed_pdbs.jsonl"
path_for_tied_positions=f"./{designsdir}/tied_pdbs.jsonl"
path_for_designed_sequences=f"./{designsdir}_mpnn/"
path_for_fixed_positions=f"./{designsdir}/fixed_pdbs.jsonl"

rfdbackbones = []
for outpdb in glob.glob(f"{designsdir}/*.pdb"):
  rfdbackbones.append(outpdb)
if len(rfdbackbones) == 0:
  raise Exception("There are no scaffold-guided RFDiffusion backbones to sequence design.")

# MspA target positions to fix sequence
fixed_positions="84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133"

run_cmd(f"python ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path={designsdir} --output_path={path_for_parsed_chains}")
run_cmd(f'python ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path={path_for_parsed_chains} --output_path={path_for_fixed_positions} --chain_list "{chains_to_design}" --position_list "{fixed_positions}"')
# Create tied positions jsonl file from homooligomer RFdiffusion outputs
run_cmd(f'python ProteinMPNN/helper_scripts/make_tied_positions_dict.py --input_path={path_for_parsed_chains} --output_path={path_for_tied_positions} --homooligomer 1')
run_cmd(f'python ProteinMPNN/protein_mpnn_run.py \
        --path_to_model_weights "./ProteinMPNN/soluble_model_weights/"\
        --jsonl_path {path_for_parsed_chains} \
        --tied_positions_jsonl {path_for_tied_positions} \
        --fixed_positions_jsonl {path_for_fixed_positions} \
        --out_folder {path_for_designed_sequences} \
        --num_seq_per_target 10 \
        --sampling_temp "0.2" \
        --batch_size 8')

alpha_1 = list("ARNDCQEGHILKMFPSTWYV-")
alpha_3 = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
           'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL','GAP']
aa_1_3 = {a:b for a,b in zip(alpha_1,alpha_3)}

def thread_mpnn_seq( pose, binder_seq ):
  rsd_set = pose.residue_type_set_for_pose( pyrosetta.rosetta.core.chemical.FULL_ATOM_t )
  for resi, mut_to in enumerate( binder_seq ):
    resi += 1 # 1 indexing
    if pose.residue(resi).name().split(':')[-1] != 'disulfide':
      name3 = aa_1_3[ mut_to ]
      new_res = pyrosetta.rosetta.core.conformation.ResidueFactory.create_residue( rsd_set.name_map( name3 ) )
      pose.replace_residue( resi, new_res, True )
  return pose

import glob
seqs = []
for fasta in glob.glob(f"{path_for_designed_sequences}/seqs/*.fa"):
  input_pdb_name = ""
  with open(fasta) as f:
    scores = []
    for l in f:
      if l.startswith('>'):
        scores = l.split()
        if input_pdb_name == "": input_pdb_name = scores[0][1:-1]
      else:
        seqs.append([input_pdb_name, scores, l.strip()])
for seq in seqs[1:]:
  idx = seq[1][1].split('=')[-1][0:-1]
  input_pdb = f"{designsdir}/{seq[0]}.pdb"
  pose = pyrosetta.pose_from_file(input_pdb)
  chainseqs = seq[2].split('/')
  singleseq = seq[2].replace('/','')
  pose = thread_mpnn_seq( pose, singleseq )
  chainlen = len(chainseqs[0])
  chains_pose = Pose()
  residue_indices = rosetta.utility.vector1_unsigned_long()
  for i in range(1,chainlen+1):
    residue_indices.append(i)
  pyrosetta.rosetta.core.pose.pdbslice(chains_pose, pose, residue_indices)
  for i in range(1,len(chainseqs)):
    start = chainlen*i+1
    residue_indices = rosetta.utility.vector1_unsigned_long()
    for j in range(start,start+chainlen):
      residue_indices.append(j)
    chainpose = Pose()
    pyrosetta.rosetta.core.pose.pdbslice(chainpose, pose, residue_indices)
    append_pose_to_pose(chains_pose, chainpose, True)

  if minimize:
    mm = MoveMap()
    mm.set_bb(False)  # Fix backbone
    mm.set_chi(True)  # Allow side chains to move
    mm.set_jump(False)
    scorefxn = create_score_function("ref2015")
    min_mover = MinMover()
    min_mover.movemap(mm)
    min_mover.score_function(scorefxn)
    print(f"minimize {path_for_designed_sequences}/{seq[0]}_{idx}.pdb")
    min_mover.apply(chains_pose)

  chains_pose.dump_pdb(f"{path_for_designed_sequences}/{seq[0]}_{idx}.pdb")
